In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import optuna # <-- NUEVO: Importar optuna
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

2025-06-16 19:23:40.222227: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750116220.732917   99815 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750116220.853655   99815 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750116224.831669   99815 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750116224.831705   99815 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750116224.831707   99815 computation_placer.cc:177] computation placer alr

In [2]:
zip_file_path = '../data/MeIA2025-Reto-01.zip'

extracted_folder_path = '../data/extracted_corpus/'

# 2. Crear la carpeta de extracción si no existe
if not os.path.exists(extracted_folder_path):
    os.makedirs(extracted_folder_path)
    print(f"Carpeta '{extracted_folder_path}' creada.")
# 3. Descomprimir archivo  
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_folder_path)
    print(f"'{zip_file_path}' descomprimido exitosamente en '{extracted_folder_path}'.")
except FileNotFoundError:
    print(f"Error: El archivo ZIP no se encontró en '{zip_file_path}'. Verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al descomprimir el archivo: {e}")
    
# 4. Listar los archivos descomprimidos (para verificar)
print("\nArchivos en la carpeta del corpus:")
corpus_files = os.listdir(extracted_folder_path)
for file_name in corpus_files:
    print(f"- {file_name}")


'../data/MeIA2025-Reto-01.zip' descomprimido exitosamente en '../data/extracted_corpus/'.

Archivos en la carpeta del corpus:
- Datos-MeIA-Reto-01


In [3]:
# Definir la ruta base donde se extrajo el contenido del ZIP
# Asegúrate de que esta ruta sea correcta relativa a tu notebook test.ipynb
# Si tu notebook está en 'notebooks/' y la extracción está en 'data/extracted_corpus/Datos-MelA-Reto-01/'
base_extracted_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/'

# Rutas completas a los archivos XLSX
train_file_path = os.path.join(base_extracted_path, 'MeIA_2025_train.xlsx')
test_file_path = os.path.join(base_extracted_path, 'MeIA_2025_test_wo_labels.xlsx')

print(f"Intentando cargar el archivo de entrenamiento desde: {train_file_path}")
print(f"Intentando cargar el archivo de prueba desde: {test_file_path}")

try:
    # Cargar el dataset de entrenamiento
    
    df_train = pd.read_excel(train_file_path)
    print(f"Datos de entrenamiento cargados correctamente")
    # Cargar el dataset de prueba (sin etiquetas)
    df_test = pd.read_excel(test_file_path)
    print(f"Datos de test cargados correctamente")


except FileNotFoundError:
    print(f"Error: Uno de los archivos XLSX no se encontró.")
    print(f"Asegúrate de que las rutas sean correctas: '{train_file_path}' y '{test_file_path}'")
    print(f"Y que la carpeta 'Datos-MelA-Reto-01' esté dentro de 'extracted_corpus'.")
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos Excel: {e}")


Intentando cargar el archivo de entrenamiento desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx
Intentando cargar el archivo de prueba desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx
Datos de entrenamiento cargados correctamente
Datos de test cargados correctamente


### Nuevo

In [8]:
# --- AJUSTE 3: Crear el Input Estructurado ---
print("Creando input estructurado...")
df_train['structured_text'] = df_train.apply(
    lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
    axis=1
)

# Renombrar columnas para el modelo
df_model = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})

# --- AJUSTE 2 (Parte 1): Preparar Label para Regresión ---
# La etiqueta debe ser float y empezar en 0.
df_model['label'] = df_model['label'].apply(lambda x: float(x) - 1.0)

# Convertir a Dataset de Hugging Face
dataset = Dataset.from_pandas(df_model[['text', 'label']])

# Dividir en entrenamiento y validación
train_test_split = dataset.train_test_split(test_size=0.2, seed=42) # Usar semilla para reproducibilidad
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

# --- 2. Cargar Modelo y Tokenizador ---
model_name = "distilbert/distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# --- AJUSTE 2 (Parte 2): Configurar Modelo para Regresión ---
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=1 # <-- ¡CLAVE! Salida única para la regresión
)

# Función de tokenización (no necesita cambios)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# --- AJUSTE 1 y 2 (Parte 3): Métrica para Regresión Ordinal ---
def compute_metrics_regression(eval_pred):
    logits, labels = eval_pred
    predictions = logits.flatten()
    
    # Redondear y limitar al rango [0, 4]
    rounded_predictions = np.round(predictions)
    clipped_predictions = np.clip(rounded_predictions, 0, 4)
    
    # Usar métrica MACRO F1
    f1 = f1_score(labels, clipped_predictions, average="macro")
    accuracy = accuracy_score(labels, clipped_predictions)
    return {"accuracy": accuracy, "f1_macro": f1}

# --- 4. Configurar Entrenamiento ---
training_args = TrainingArguments(
    output_dir="./results_beto_ordinal_v2",
    num_train_epochs=4, # Podemos probar con un poco más de épocas
    per_device_train_batch_size=8, # Bajar si hay problemas de memoria
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2, # Simula un batch size de 16 (8*2)
    warmup_ratio=0.1, # Usar warmup_ratio es más flexible que steps fijos
    weight_decay=0.01,
    logging_dir='./logs_beto_ordinal_v2',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro", # ¡Importante!
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_regression,
)

# ¡A entrenar!
print("Iniciando entrenamiento con el modelo ordinal y texto estructurado...")
trainer.train()

Creando input estructurado...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_89132/1161161886.py:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Iniciando entrenamiento con el modelo ordinal y texto estructurado...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.921200,1.076977,0.339000,0.300542
2,0.634200,0.721929,0.484000,0.475630
3,0.389400,0.733024,0.471000,0.464033
4,0.242500,0.695255,0.475000,0.473811


TrainOutput(global_step=1000, training_loss=0.7794581022262573, metrics={'train_runtime': 604.9842, 'train_samples_per_second': 26.447, 'train_steps_per_second': 1.653, 'total_flos': 2119440580608000.0, 'train_loss': 0.7794581022262573, 'epoch': 4.0})

### Viejo

In [4]:
# --- Asumimos que tu DataFrame 'df_train' ya está cargado ---

# Preparar el DataFrame
df_train_tabularisai = df_train.rename(columns={'Review': 'text', 'Polarity': 'label'})
df_train_tabularisai['label'] = df_train_tabularisai['label'].apply(lambda x: int(x) - 1)

dataset = Dataset.from_pandas(df_train_tabularisai)
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

# Cargar el tokenizador y el modelo que seleccionaste
model_name = "tabularisai/multilingual-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

training_args = TrainingArguments(
    output_dir="./results_tabularisai",
    num_train_epochs=3,
    learning_rate=2e-5, # Un learning rate bajo es bueno para el fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs_tabularisai',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# Iniciar el afinamiento
trainer.train()

print("¡Afinamiento del modelo de tabularisai completado!")

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.025400,1.087794,0.534000,0.537266
2,0.881900,1.111234,0.528000,0.528628
3,0.771000,1.154419,0.528000,0.530826


¡Afinamiento del modelo de tabularisai completado!


In [5]:
# Ruta a la carpeta donde quieres guardar el modelo
output_dir = "./modelos/tabularisai"

# Guardar el modelo y el tokenizador en esa carpeta
trainer.save_model(output_dir)

### bert-base-multilingual-uncased-sentiment 

In [4]:
# Preparar el DataFrame
df_train_nlptown = df_train.rename(columns={'Review': 'text', 'Polarity': 'label'})
df_train_nlptown['label'] = df_train_nlptown['label'].apply(lambda x: int(x) - 1) # Ajustamos a 0-4

# Convertir a Dataset y dividir
dataset = Dataset.from_pandas(df_train_nlptown)
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']


# Cargar el tokenizador y el modelo que seleccionaste
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
# Las etiquetas en este modelo son '1 star' a '5 stars', pero el Trainer las manejará internamente
# si le pasamos num_labels=5 y las etiquetas de 0 a 4.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

training_args = TrainingArguments(
    output_dir="./results_nlptown",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5, # Un learning rate más bajo suele ser bueno para el fine-tuning
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs_nlptown',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# Iniciar el afinamiento
trainer.train()

print("¡Afinamiento del modelo de nlptown completado!")

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.048400,1.104344,0.484000,0.477437
2,0.816600,1.077669,0.528000,0.524725
3,0.664300,1.195405,0.544000,0.543270


¡Afinamiento del modelo de nlptown completado!


In [5]:
# Ruta a la carpeta donde quieres guardar el modelo
output_dir = "./modelos/bert-sentiment"

# Guardar el modelo y el tokenizador en esa carpeta
trainer.save_model(output_dir)

### dccuchile/bert-base-spanish-wwm-cased

In [4]:
print("Creando input estructurado...")
df_train['structured_text'] = df_train.apply(
    lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
    axis=1
)


Creando input estructurado...


In [17]:
# Paso 2: Preparar el DataFrame y convertirlo a un Dataset de Hugging Face
# Renombramos las columnas y ajustamos las etiquetas (si no lo has hecho en el df original)
df_train_beto = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})
df_train_beto['label'] = df_train_beto['label'].apply(lambda x: int(x) - 1)

# Convertir a Dataset
dataset = Dataset.from_pandas(df_train_beto)

# Dividir en entrenamiento y validación
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']


# Paso 3: Cargar el tokenizador y el modelo BETO
# ¡Este es el cambio principal!
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5) # 5 clases de polaridad

# Paso 4: Tokenizar los datos
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Paso 5: Definir la métrica de evaluación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

# Paso 6: Configurar y ejecutar el entrenamiento
training_args = TrainingArguments(
    output_dir="./results_beto",     # Nuevo directorio para no sobreescribir el anterior
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_beto',       # Nuevo directorio de logs
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# ¡Iniciar el entrenamiento!
trainer.train()

print("¡Entrenamiento con BETO completado!")

# Para guardar el modelo final y el tokenizador
# trainer.save_model("./mi_modelo_beto_final")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.118000,1.042671,0.520000,0.478475
2,1.039900,1.035133,0.518000,0.456780
3,0.536400,1.021053,0.568000,0.570595


¡Entrenamiento con BETO completado!


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

[I 2025-06-16 19:24:13,538] A new study created in memory with name: no-name-f99b45b1-543d-4796-9bef-7a42de3ec7c6


Iniciando la búsqueda de hiperparámetros para el modelo de CLASIFICACIÓN...


/tmp/ipykernel_99815/3395898629.py:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

{'eval_loss': 1.0875511169433594, 'eval_accuracy': 0.536, 'eval_f1_weighted': 0.5388378741338824, 'eval_f1_macro': 0.5331159068675333, 'eval_runtime': 15.6161, 'eval_samples_per_second': 64.036, 'eval_steps_per_second': 8.005, 'epoch': 3.0}


/tmp/ipykernel_99815/3395898629.py:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 1.2562, 'grad_norm': 11.148843765258789, 'learning_rate': 8.699229286856916e-06, 'epoch': 1.0}
{'eval_loss': 1.0300229787826538, 'eval_accuracy': 0.528, 'eval_f1_weighted': 0.5263121897600513, 'eval_f1_macro': 0.5180660208769025, 'eval_runtime': 15.6659, 'eval_samples_per_second': 63.833, 'eval_steps_per_second': 7.979, 'epoch': 1.0}
{'loss': 0.921, 'grad_norm': 13.11844539642334, 'learning_rate': 5.801418065385591e-06, 'epoch': 2.0}
{'eval_loss': 1.0555680990219116, 'eval_accuracy': 0.526, 'eval_f1_weighted': 0.5286289242523485, 'eval_f1_macro': 0.5231242763009155, 'eval_runtime': 18.6468, 'eval_samples_per_second': 53.628, 'eval_steps_per_second': 6.704, 'epoch': 2.0}


[W 2025-06-16 19:59:23,969] Trial 2 failed with parameters: {'learning_rate': 1.0356777305538514e-05, 'num_train_epochs': 4, 'per_device_train_batch_size': 8, 'warmup_steps': 213, 'weight_decay': 0.05091543385047033} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/joad/Personal/Desafio/Desafio_NLP_UNAM/venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_99815/3395898629.py", line 73, in objective
    trainer.train()
  File "/home/joad/Personal/Desafio/Desafio_NLP_UNAM/venv/lib/python3.12/site-packages/transformers/trainer.py", line 2240, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/joad/Personal/Desafio/Desafio_NLP_UNAM/venv/lib/python3.12/site-packages/transformers/trainer.py", line 2656, in _inner_training_loop
    self._maybe_log_save_evaluate(
  File "/home/joad/Pe

KeyboardInterrupt: 

In [18]:
# Ruta a la carpeta donde quieres guardar el modelo
output_dir = "./modelos/bert"

# Guardar el modelo y el tokenizador en esa carpeta
trainer.save_model(output_dir)

### Roberta-base-bne

In [10]:
df=df_train.copy()

In [11]:
# Paso 2: Preparar el DataFrame y convertirlo a un Dataset de Hugging Face
# Renombraremos las columnas para mayor claridad y estándar.
df = df.rename(columns={'Review': 'text', 'Polarity': 'label'})

# IMPORTANTE: Los modelos de clasificación de Hugging Face esperan que las etiquetas
# comiencen en 0. Tus etiquetas son (1.0, 2.0, 3.0, 4.0, 5.0).
# Las ajustaremos para que sean (0, 1, 2, 3, 4).
df['label'] = df['label'].apply(lambda x: int(x) - 1)

# Convertir el DataFrame de pandas a un Dataset de Hugging Face
dataset = Dataset.from_pandas(df)

# (Opcional pero recomendado) Dividir el dataset en entrenamiento y validación
# para monitorear el rendimiento durante el entrenamiento.
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']


# Paso 3: Cargar el tokenizador y el modelo
model_name = "PlanTL-GOB-ES/roberta-base-bne"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5) # 5 clases de polaridad (0 a 4)

# Paso 4: Tokenizar los datos
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Paso 5: Definir la métrica de evaluación
# La métrica del reto es un F1 ponderado, aquí definimos cómo calcularla.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Usaremos 'weighted' para manejar el posible desbalance de clases.
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

# Paso 6: Configurar y ejecutar el entrenamiento
training_args = TrainingArguments(
    output_dir="./results",          # Directorio donde se guardarán los resultados y el modelo
    num_train_epochs=3,              # Número de épocas (puedes ajustar esto)
    per_device_train_batch_size=16,  # Tamaño del lote por dispositivo durante el entrenamiento
    per_device_eval_batch_size=16,   # Tamaño del lote para la evaluación
    warmup_steps=500,                # Número de pasos de calentamiento para el scheduler de learning rate
    weight_decay=0.01,               # Fuerza de la regularización
    logging_dir='./logs',            # Directorio para los logs
    logging_steps=10,
    eval_strategy="epoch",     # La evaluación se realiza al final de cada época
    save_strategy="epoch",           # El modelo se guarda al final de cada época
    load_best_model_at_end=True,     # Cargar el mejor modelo al final del entrenamiento
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# ¡Iniciar el entrenamiento!
trainer.train()

print("¡Entrenamiento completado!")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-bne and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.063200,1.019214,0.538000,0.536988
2,1.046500,1.153523,0.488000,0.457376
3,0.514300,1.111596,0.570000,0.569992


¡Entrenamiento completado!


In [15]:
# Ruta a la carpeta donde quieres guardar el modelo
output_dir = "./modelos/roberta"

# Guardar el modelo y el tokenizador en esa carpeta
trainer.save_model(output_dir)

In [13]:

# Paso 1: Realizar la predicción en el conjunto de evaluación
print("Realizando predicciones en el conjunto de evaluación...")
predictions_output = trainer.predict(tokenized_eval_dataset)

# El objeto 'predictions_output' contiene las predicciones (logits), las etiquetas reales y las métricas.
print(predictions_output.metrics)

# Paso 2: Extraer las predicciones y las etiquetas verdaderas
# predictions_output.predictions es un array con los logits (valores de salida brutos para cada clase)
# Usamos np.argmax para obtener el índice de la clase con el mayor logit para cada reseña.
y_pred_indices = np.argmax(predictions_output.predictions, axis=1)

# predictions_output.label_ids contiene los índices de las etiquetas verdaderas
y_true_indices = predictions_output.label_ids

# Paso 3: Convertir los índices predichos a las etiquetas originales (1-5)
# Recuerda que restamos 1 para el entrenamiento (0-4), ahora sumamos 1 para volver al formato original (1-5)
y_pred_labels = [i + 1 for i in y_pred_indices]
y_true_labels = [i + 1 for i in y_true_indices]

# Paso 4: Crear un DataFrame para comparar los resultados
# Usaremos el dataset de evaluación original (antes de tokenizar) para obtener el texto.
# El eval_dataset tiene las columnas 'text' y 'label' (con etiquetas 0-4)
df_results = pd.DataFrame({
    "Review": eval_dataset["text"],
    "Polarity_Real": y_true_labels,
    "Polarity_Predicha": y_pred_labels
})

# # Mostrar las primeras 20 predicciones para ver cómo se comportó el modelo
# print("\nComparación de predicciones vs. etiquetas reales:")
# print(df_results.head(20))

# (Opcional) Ver los casos donde el modelo se equivocó
print("\nEjemplos donde el modelo se equivocó:")
df_errors = df_results[df_results["Polarity_Real"] != df_results["Polarity_Predicha"]]
df_errors.head(10)

Realizando predicciones en el conjunto de evaluación...


{'test_loss': 1.0192135572433472, 'test_accuracy': 0.538, 'test_f1_weighted': 0.5369875288153076, 'test_runtime': 7.4272, 'test_samples_per_second': 67.32, 'test_steps_per_second': 4.308}

Ejemplos donde el modelo se equivocó:


,Review,Polarity_Real,Polarity_Predicha
0,"Me encanto el hotel y sus servicios, todo esta...",3,4
2,La comida era buena. Es el muy pequeño plato d...,3,2
3,"Ayer, despues de un viaje largo en coche yo y ...",1,2
4,Hemos estado yendo a Posada del Mar por más de...,5,2
5,El restaurante tiene un ambiente agradable y l...,4,3
7,Pasamos dos semanas en Ajijic último año y enc...,5,4
8,Fue un agradable y hermoso lugar. Tiene una pl...,3,4
10,Nuestra guía era Arturo Cárdenas y era diverti...,4,3
11,"La atención es muy buena, moderada y cordial. ...",3,4
12,el burrito de arrachera viene atascado de arro...,2,1


### 3re modelos

### Test

In [ ]:
df=df_train.copy()

df = df.rename(columns={'Polarity': 'labels'})
df['labels'] = df['labels'] - 1
df['labels'] = df['labels'].astype(int)

In [ ]:

# --- División de datos (tu código original) ---
df_train_split, df_eval_split = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['labels']
)

# --- Tokenización (tu código original) ---
model_checkpoint = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

hf_train_dataset = Dataset.from_pandas(df_train_split[['Review', 'labels']])
hf_eval_dataset = Dataset.from_pandas(df_eval_split[['Review', 'labels']])

def tokenize_function(examples):
    return tokenizer(examples["Review"], truncation=True, padding="max_length", max_length=128)

tokenized_train_dataset = hf_train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset  = hf_eval_dataset.map(tokenize_function, batched=True)

# --- Métrica de Evaluación (tu código original) ---
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    # IMPORTANTE: La métrica del reto es 'weighted' F1
    f1 = f1_score(labels, predictions, average="weighted") 
    return {"f1": f1}

# --- NUEVO: Función para inicializar el modelo ---
# Optuna necesita una función que devuelva un modelo nuevo en cada trial
# para asegurar que no haya "fugas" de pesos entre entrenamientos.
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=5)

# --- MODIFICADO: Argumentos de Entrenamiento ---
# Estos son los argumentos base. Los que se van a tunear los definiremos en el espacio de búsqueda.
training_args = TrainingArguments(
    output_dir="./results_optuna",
    logging_dir="./logs_optuna",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # La métrica ahora debe coincidir con la que devuelve compute_metrics
    metric_for_best_model="f1", 
    greater_is_better=True,
    fp16=True if torch.cuda.is_available() else False,
)

# --- MODIFICADO: Inicialización del Trainer ---
# Ya no pasamos un modelo, sino la función que lo inicializa.
trainer = Trainer(
    model_init=model_init, # <-- Usamos model_init en lugar de model=...
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# --- NUEVO: Definir el espacio de búsqueda de hiperparámetros para Optuna ---
def hp_space_optuna(trial: optuna.Trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 1, 4),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 1000),
    }

# --- NUEVO Y FINAL: Ejecutar la búsqueda de hiperparámetros ---
print("\nIniciando la búsqueda de hiperparámetros con Optuna...")
best_run = trainer.hyperparameter_search(
    direction="maximize",      # Queremos maximizar el F1-score
    backend="optuna",          # Usamos el backend de Optuna
    hp_space=hp_space_optuna,  # La función que define qué tunear
    n_trials=5,               # Número de combinaciones a probar (puedes aumentarlo)
    
)

print("\n¡Búsqueda completada!")
print(f"Mejor trial encontrado:")
print(f"  -> Valor de la métrica (f1): {best_run.objective}")
print(f"  -> Mejores Hiperparámetros: {best_run.hyperparameters}")

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_67693/1276984100.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


RuntimeError: Error(s) in loading state_dict for Linear:
	size mismatch for bias: copying a param with shape torch.Size([20]) from checkpoint, the shape in current model is torch.Size([5]).

In [11]:
# --- RE-ENTRENAMIENTO DEL MODELO FINAL CON LOS MEJORES HIPERPARÁMETROS ---

print("\nRe-entrenando el modelo final con los mejores hiperparámetros...")

# 1. Obtén los mejores hiperparámetros del objeto best_run
best_hyperparameters = best_run.hyperparameters

# 2. Crea nuevos TrainingArguments, actualizándolos con los mejores valores
#    Es una buena práctica guardar el modelo final en su propio directorio.
final_training_args = TrainingArguments(
    output_dir="./results_final_model",
    logging_dir="./logs_final_model",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True if torch.cuda.is_available() else False,
    seed=42,
    # ** Actualizamos con los mejores hiperparámetros encontrados **
    learning_rate=best_hyperparameters['learning_rate'],
    num_train_epochs=best_hyperparameters['num_train_epochs'],
    per_device_train_batch_size=best_hyperparameters['per_device_train_batch_size'],
    weight_decay=best_hyperparameters['weight_decay'],
    warmup_steps=best_hyperparameters['warmup_steps'],
)

# 3. Inicializa un nuevo Trainer para el entrenamiento final.
#    Esta vez, pasamos una instancia del modelo directamente con `model`, no `model_init`.
final_trainer = Trainer(
    model=model_init(),  # Llama a la función para obtener un modelo nuevo
    args=final_training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 4. Entrena el modelo final
final_trainer.train()

# 5. ¡Ahora sí! Realiza las predicciones con el trainer final
print("\nRealizando predicciones en el conjunto de evaluación con el modelo final...")
predictions_output = final_trainer.predict(tokenized_eval_dataset)

# --- ANÁLISIS DE RESULTADOS (tu código original, sin cambios) ---

# Las predicciones son logits, convertirlos a etiquetas de clase
predicted_labels = np.argmax(predictions_output.predictions, axis=1)

# Las etiquetas reales (ground truth)
true_labels = predictions_output.label_ids

textos_validacion = df_eval_split['Review'].reset_index(drop=True)

# Creamos el DataFrame
df_resultados = pd.DataFrame({
    'Review': textos_validacion,
    'Etiqueta_Verdadera': true_labels,
    'Prediccion_Modelo': predicted_labels
})



Re-entrenando el modelo final con los mejores hiperparámetros...


/tmp/ipykernel_56794/1780600650.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  final_trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,1.060000,1.005725,0.547778
2,0.981000,1.008334,0.554506



Realizando predicciones en el conjunto de evaluación con el modelo final...


In [12]:
# Contar cuántas predicciones fueron correctas
predicciones_correctas = (df_resultados['Etiqueta_Verdadera'] == df_resultados['Prediccion_Modelo']).sum()

# Obtener el número total de predicciones
total_predicciones = len(df_resultados)

# Calcular el porcentaje de acierto
porcentaje_acierto = (predicciones_correctas / total_predicciones) * 100

print(f"\n--- Resultados de Acierto ---")
print(f"Predicciones Correctas: {predicciones_correctas}")
print(f"Total de Muestras de Validación: {total_predicciones}")
print(f"Porcentaje de Acierto (Accuracy): {porcentaje_acierto:.2f}%")


--- Resultados de Acierto ---
Predicciones Correctas: 556
Total de Muestras de Validación: 1000
Porcentaje de Acierto (Accuracy): 55.60%
